In [1]:
from pathlib import Path
import json
OUT=(next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p/'dataset_2.csv').is_file()) / 'fusion/cnn_bilstm_v1');nb=json.loads((OUT/'01_fusion_training.ipynb').read_text(encoding='utf-8'));exec(compile(''.join(nb['cells'][0]['source']),'common_definitions_only','exec'))
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
assert not (OUT/'final_fusion_report.md').exists()
cks={h:torch.load(OUT/'models'/f'fusion_{h}s_best.pt',map_location='cpu',weights_only=False) for h in [1,2,3]}
assert all(sha(Path(p))==v for p,v in cks[1]['input_hashes'].items())
cref=json.loads((CR/'codex_performance_audit/cnn_performance_audit_results.json').read_text(encoding='utf-8'))['common_cohort'];bref=json.loads((BR/'codex_independent_audit/audit_results.json').read_text(encoding='utf-8'))['horizons'];results={};plotdata={}
def paired(y,target,fp,fm,bp,bm):
 rng=np.random.default_rng(2026);eq=np.flatnonzero(y==1);noise=np.flatnonzero(y==0);fdelta=[];mdelta=[]
 def f1fast(yy,p):
  yy=yy==1;q=p>=.5;tp=(yy&q).sum();return float(2*tp/(2*tp+(~yy&q).sum()+(yy&~q).sum()))
 for _ in range(500):
  a=rng.choice(eq,len(eq),replace=True);n=rng.choice(noise,len(noise),replace=True);ix=np.r_[a,n];fdelta.append(f1fast(y[ix],fp[ix])-f1fast(y[ix],bp[ix]));mdelta.append(float((np.abs(fm[a]-target[a])-np.abs(bm[a]-target[a])).mean()))
 return {'replicates':500,'seed':2026,'method':'paired stratified bootstrap; same sampled EQ/noise indices for both models','f1_difference_ci95':np.quantile(fdelta,[.025,.975]).tolist(),'mae_difference_ci95':np.quantile(mdelta,[.025,.975]).tolist()}


Alignment PASS; 64-dimensional learned embedding concatenation; frozen backbones.


In [2]:
for h,ck in cks.items():
 c,b,cc,bc=branches(h);head=FusionHead().to(DEV);head.load_state_dict(ck['head_state'],strict=True);head.eval();mu=ck['embedding_mean'];sd=ck['embedding_scale']
 va=encode('validation',h,c,b,cc,bc);vl=epoch(head,batches(va,mu,sd),ck['target_center'],ck['target_scale']);assert abs(vl['total']-ck['best_validation_loss'])<1e-6
 d=encode('test',h,c,b,cc,bc,only_common=True);y=d['y'].numpy();target=d['m'].numpy();reg=d['reg'].numpy();identity=hashlib.sha256('\n'.join(d['ids']).encode()).hexdigest();assert identity==cref['identity_sha256'] and int(reg.sum())==1200 and int((~reg).sum())==1214
 p,z=predict(head,d['z'],mu,sd);m=z*ck['target_scale']+ck['target_center'];models={}
 for name,prob,pred in [('CNN',d['cnn_p'],d['cnn_m']),('BiLSTM',d['bilstm_p'],d['bilstm_m']),('Fusion',p,m)]:models[name]={'fixed_05':det(y,prob,.5),'magnitude':magnitude(target[reg],pred[reg])}
 # Verify baseline eligibility/threshold results, without repeating standalone training/audits.
 for name,ref in [('CNN',cref['metrics'][str(h)]),('BiLSTM',bref[str(h)]['common']['fixed_05_detection'])]:
  for key in ['accuracy','precision','recall','f1']:assert abs(models[name]['fixed_05'][key]-ref[key])<1e-6,f'Baseline policy mismatch {name}/{h}/{key}'
 assert abs(models['CNN']['magnitude']['mae']-cref['metrics'][str(h)]['mae'])<1e-6 and abs(models['BiLSTM']['magnitude']['mae']-bref[str(h)]['common']['magnitude']['mae'])<1e-6
 comparison={}
 for name,bp,bm in [('CNN',d['cnn_p'],d['cnn_m']),('BiLSTM',d['bilstm_p'],d['bilstm_m'])]:
  comparison[name]={'fusion_minus_baseline_f1':models['Fusion']['fixed_05']['f1']-models[name]['fixed_05']['f1'],'fusion_minus_baseline_mae':models['Fusion']['magnitude']['mae']-models[name]['magnitude']['mae'],**paired(y,target,p,m,bp,bm)}
 results[h]={'common_identity_sha256':identity,'n':len(y),'earthquakes':int(reg.sum()),'noise':int((~reg).sum()),'models':models,'fusion_validation_threshold':ck['threshold'],'fusion_selected':det(y,p,ck['threshold']),'fusion_low_fpr_threshold':ck['low_fpr_threshold'],'fusion_low_fpr_validation':ck['low_fpr_validation'],'fusion_low_fpr_test':det(y,p,ck['low_fpr_threshold']),'paired_comparisons':comparison,'baseline_threshold_policy':'fixed .5; selected fusion and low-FPR metrics reported separately','noise_family_shortcuts':'UNKNOWN; not established by this experiment'}
 plotdata[h]={'actual':target[reg],'predicted':m[reg],'residual':m[reg]-target[reg]}
 print(f'{h}s common fusion: F1(.5)={models["Fusion"]["fixed_05"]["f1"]:.4f}, MAE={models["Fusion"]["magnitude"]["mae"]:.4f}',flush=True)


1s common fusion: F1(.5)=0.6799, MAE=0.6642
2s common fusion: F1(.5)=0.7163, MAE=0.6359
3s common fusion: F1(.5)=0.7475, MAE=0.6082


In [3]:
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
def save(fig,name):
 p=OUT/'plots'/name;assert not p.exists();fig.tight_layout();fig.savefig(p,dpi=300,bbox_inches='tight');plt.close(fig)
fig,axs=plt.subplots(1,3,figsize=(14,4))
for ax,(h,ck) in zip(axs,cks.items()):
 e=[q['epoch'] for q in ck['history']];ax.plot(e,[q['train']['total'] for q in ck['history']],label='Train epoch-end');ax.plot(e,[q['validation']['total'] for q in ck['history']],label='Validation');ax.axvline(ck['best_epoch'],color='black',ls=':');ax.set(title=f'{h}s',xlabel='Epoch',ylabel='BCE + masked standardized Huber');ax.legend()
save(fig,'training_validation_loss.png')
fig,axs=plt.subplots(1,3,figsize=(12,4))
for ax,(h,r) in zip(axs,results.items()):
 cm=np.array(r['models']['Fusion']['fixed_05']['confusion_matrix']);ax.imshow(cm,cmap='Blues');ax.set(title=f'{h}s fusion; threshold .5',xticks=[0,1],yticks=[0,1],xticklabels=['Noise','EQ'],yticklabels=['Noise','EQ'],xlabel='Predicted',ylabel='Actual')
 for i in range(2):
  for j in range(2):ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black')
save(fig,'confusion_matrices.png')
fig,axs=plt.subplots(1,3,figsize=(14,4))
for ax,(h,d) in zip(axs,plotdata.items()):
 ax.scatter(d['actual'],d['predicted'],s=9,alpha=.35);lo=min(d['actual'].min(),d['predicted'].min());hi=max(d['actual'].max(),d['predicted'].max());ax.plot([lo,hi],[lo,hi],'k--');ax.set(title=f'{h}s common EQ',xlabel='Actual magnitude',ylabel='Predicted magnitude')
save(fig,'actual_predicted_magnitude.png')
fig,axs=plt.subplots(1,3,figsize=(14,4))
for ax,(h,d) in zip(axs,plotdata.items()):ax.hist(d['residual'],bins=35);ax.axvline(0,color='black',ls='--');ax.set(title=f'{h}s',xlabel='Predicted minus actual',ylabel='Count')
save(fig,'magnitude_residuals.png')
for metric,name in [('f1','three_model_f1.png'),('mae','three_model_mae.png')]:
 fig,ax=plt.subplots(figsize=(8,4));x=np.arange(3)
 for offset,model in zip([-.25,0,.25],['CNN','BiLSTM','Fusion']):ax.bar(x+offset,[results[h]['models'][model]['fixed_05' if metric=='f1' else 'magnitude'][metric] for h in [1,2,3]],width=.25,label=model)
 ax.set(xticks=x,xticklabels=['1s','2s','3s'],xlabel='Observed post-P horizon',ylabel=metric.upper(),title='Identical common test cohort'+('; threshold .5' if metric=='f1' else ''));ax.legend();save(fig,name)
assert all(sha(Path(p))==v for p,v in cks[1]['input_hashes'].items())
lines=['# CNN + BiLSTM fusion v1','', '**Date:** 2026-11-13. Three heads trained and evaluated; no Attention.','', '## Architecture and fixed protocol','', 'Exact standalone architecture classes imported from their notebook ASTs without executing original training code. Each horizon loads its own best CNN/BiLSTM weights, frozen with requires_grad=False and eval mode (including backbone dropout). Forward hooks capture each 32-dimensional shared/penultimate representation. Only these learned embeddings are concatenated, not raw inputs or standalone predictions. Embeddings are cached in memory, never saved as a duplicate dataset.','', 'The 64-dimensional concatenation is standardized by eligible training-embedding means/SD (SD<=1e-8 uses 1); frozen transforms apply to validation/test. Fusion MLP: Linear(64,32), SiLU, dropout .2, then separate binary-logit and magnitude heads. AdamW .001, weight_decay .0001; BCE plus masked Huber(delta1) with train-only eligible magnitude mean/SD; gradient norm5; seed42; maximum35 epochs, patience7, minimum validation improvement .0001. Validation criterion is full-set mean BCE plus eligible-EQ mean Huber. No noise magnitude sentinel enters regression.','', 'Fusion threshold policy: validation-only F1 grid .10:.01:.90, tie closest .5 then lower. Low-FPR operating point prespecified as <=5% empirical validation noise FPR, using the negative-score order-statistic boundary; ties can make it conservative. Test FPR is measured, not guaranteed <=5%. Both policies frozen before test. Fixed .5 is the only direct baseline threshold comparison.','',f"Trainable parameters per head: {cks[1]['trainable_parameters']}. Environment: {cks[1]['environment']}.",'', '## Mandatory alignment','',f"Parent-ID keyed join with exact source/pair/combined ID, labels, magnitudes and all horizon eligibility/regression-mask equality: `{json.dumps(ALIGNMENT)}`. Cross-split parent intersections zero. Sampling100Hz, shared100-sample pre-context, waveform200/300/400 samples, tabular20/30/40 bins verified. Tabular records are explicitly reordered by parent lookup, even where resulting order is identical. Alignment failure raises before training. No arrival values, true magnitude, identity or masks are network input features.",'', '## Selected fusion checkpoints','', '| Horizon | Best / run epochs | Validation loss | F1 threshold | Low-FPR threshold | Fit seconds | Checkpoint |','|---|---:|---:|---:|---:|---:|---|']
for h,ck in cks.items():lines.append(f"| {h}s | {ck['best_epoch']} / {len(ck['history'])} | {ck['best_validation_loss']:.7f} | {ck['threshold']:.2f} | {ck['low_fpr_threshold']:.9f} | {ck['training_seconds']:.2f} | models/fusion_{h}s_best.pt |")
lines+=['','Best head weights only are saved, with frozen-backbone paths/hashes, embedding scalers, target transforms, thresholds, histories and configuration. Standalone weights are not duplicated in these small checkpoints; reconstruct using the exact referenced source checkpoints. Weights checked tensor-exact after training and all six original checkpoint file hashes checked before/after.','', '## Identical common-cohort results','', 'All models evaluated on the same 1,200 earthquakes and 1,214 noise records, with ordered source-ID hash matching the audited CNN. Regression covers all 1,200 eligible earthquakes, independent of detection success. Baseline fixed-.5 accuracy/precision/recall/F1 and magnitude MAE reproduced within1e-6. No standalone retraining or full audit repeated.','', '| Horizon | Model | Accuracy | Precision | Recall | F1 | FPR | ROC-AUC | PR-AUC (AP) | MAE | RMSE | R2 |','|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|']
for h,r in results.items():
 for name,v in r['models'].items():
  d=v['fixed_05'];m=v['magnitude'];lines.append(f'| {h}s | {name} | '+' | '.join(f'{d[k]:.6f}' for k in ['accuracy','precision','recall','f1','false_positive_rate','roc_auc','pr_auc_ap'])+' | '+' | '.join(f'{m[k]:.6f}' for k in ['mae','rmse','r2'])+' |')
lines+=['','## Fusion operating points','','| Horizon | Policy | Threshold | Precision | Recall | F1 | Test FPR | Confusion matrix [noise,EQ] |','|---|---|---:|---:|---:|---:|---:|---|']
for h,r in results.items():
 for policy,t,d in [('Fixed',.5,r['models']['Fusion']['fixed_05']),('Validation F1',r['fusion_validation_threshold'],r['fusion_selected']),('Validation FPR<=.05',r['fusion_low_fpr_threshold'],r['fusion_low_fpr_test'])]:lines.append(f"| {h}s | {policy} | {t:.9f} | {d['precision']:.6f} | {d['recall']:.6f} | {d['f1']:.6f} | {d['false_positive_rate']:.6f} | {d['confusion_matrix']} |")
 lines.append(f"\n{h}s low-FPR validation precision/recall/FPR: {r['fusion_low_fpr_validation']['precision']:.6f} / {r['fusion_low_fpr_validation']['recall']:.6f} / {r['fusion_low_fpr_validation']['false_positive_rate']:.6f}.\n")
lines+=['','## Paired evidence and bias','','Differences below are fusion minus baseline, at fixed .5 for F1, same EQ targets for MAE. 500 paired stratified bootstrap resamples, seed2026, provide descriptive percentile95% intervals. Negative MAE difference and positive F1 difference favor fusion; intervals crossing zero do not establish a robust gain. One training seed, reused exploratory test cohort, and no multiple-comparison adjustment: no general or operational superiority claim.','', '| Horizon | Baseline | F1 difference [CI95] | MAE difference [CI95] |','|---|---|---|---|']
for h,r in results.items():
 for name,d in r['paired_comparisons'].items():lines.append(f"| {h}s | {name} | {d['fusion_minus_baseline_f1']:.6f} {d['f1_difference_ci95']} | {d['fusion_minus_baseline_mae']:.6f} {d['mae_difference_ci95']} |")
lines+=['','| Horizon | Fusion bias | Pred/actual SD | Slope | Actual/predicted variance | M>=5 N | Underprediction % | Strong bias |','|---|---:|---:|---:|---|---:|---:|---:|']
for h,r in results.items():
 m=r['models']['Fusion']['magnitude'];lines.append(f"| {h}s | {m['bias']:.6f} | {m['std_ratio']:.5f} | {m['slope']:.5f} | {m['actual_variance']:.6f}/{m['predicted_variance']:.6f} | {m['strong_n']} | {m['strong_underprediction_percent']:.3f} | {m['strong_bias']:.6f} |")
lines+=['','## Scientific limitations','','Both branches derive from the same synthetic signal source; tabular summaries are not independent sensor observations. Oracle P/reference alignment is assumed. Eligible S masks define offline research populations, not a deployed decision rule. Operational EEW latency, autonomous picking and warning lead time remain unverified; no runtime claim is inferred from head-training speed. Noise-family shortcut status is UNKNOWN, and fusion does not prove removal of known background risks. Magnitude compression, strong-event underprediction and high false-positive rates remain risks quantified above. Differences include representation/capacity/budget effects and cannot isolate architecture alone.','', 'The test set has already been inspected in earlier research. Architecture, budgets, thresholds and checkpoints here were chosen using only training/validation; test paired comparisons are descriptive and did not trigger tuning. All fusion checkpoint validation losses reproduced after reload within1e-6. No Attention or further model change follows this evaluation.','', '## Outputs and preservation','']
for name in ['training_validation_loss.png','confusion_matrices.png','actual_predicted_magnitude.png','magnitude_residuals.png','three_model_f1.png','three_model_mae.png']:lines.append(f'- [plots/{name}](plots/{name}) — PNG at300 DPI')
lines+=['',f"All {len(cks[1]['input_hashes'])} original dependencies hashed unchanged, including all six standalone checkpoints, both datasets and original reports/notebooks. Writes confined to fusion/cnn_bilstm_v1. No waveforms/features regenerated, no full datasets copied, no existing file modified/deleted/moved. Complete hashes are embedded in each fusion checkpoint.",'', 'Complete evaluation values, operating points and paired intervals:','', '```json',json.dumps(results,indent=2),'```','', '**STOP after fusion v1. Independent audit required before Attention.**']
with (OUT/'final_fusion_report.md').open('x',encoding='utf-8') as f:f.write('\n'.join(lines)+'\n')
print('Six essential plots and final report saved; all original dependencies unchanged.')


Six essential plots and final report saved; all original dependencies unchanged.


In [4]:
# Interpret already saved paired comparisons; no inference or tuning.
from pathlib import Path
p=(next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p/'dataset_2.csv').is_file()) / 'fusion/cnn_bilstm_v1/final_fusion_report.md')
text=p.read_text(encoding="utf-8")
summary="""## Observed outcome

Fusion does not show a consistent advantage over the frozen CNN. Fixed-.5 F1 is lower at 1s and 3s; the small 2s F1 increase has a paired interval crossing zero. All fusion-versus-CNN MAE intervals cross zero, including the 3s point estimate improvement. Compared with BiLSTM, fixed-.5 F1 is worse at 1s and better at 2s/3s in the descriptive paired bootstrap; magnitude gains remain uncertain. These are exploratory, single-seed comparisons, not confirmatory superiority findings.

Validation-defined low-FPR thresholds yield test FPR 4.20%, 4.53%, 4.70% with recall 23.33%, 46.50%, 57.42% at 1s/2s/3s. This operating-point tradeoff should not be conflated with fixed-.5 comparisons. Magnitude predictions remain compressed (SD ratios 0.396–0.570) and underpredict 92.37–98.47% of magnitude>=5 events. No settings or thresholds were changed after these findings.

"""
anchor="## Scientific limitations"
assert anchor in text
p.write_text(text.replace(anchor,summary+anchor,1),encoding="utf-8")
print("Recorded mixed fusion outcomes and paired uncertainty; no additional training or inference.")


Recorded mixed fusion outcomes and paired uncertainty; no additional training or inference.
